#### This notebook is for logistic regression

In [24]:
library(rsample)     # data splitting 
library(dplyr)       # data wrangling
library(rpart)       # performing regression trees
library(rpart.plot)  # plotting regression trees
library(ipred)       # bagging
library(caret)       # bagging
library(smotefamily)
library(janitor) 
library(readr)
library(tidyverse)
library(caret)
library(xgboost)
library(glmnet)
library(here)
setwd(here::here())

In [25]:
# DATA RELATED PARAMS
def_outbreak = 4 # number of cases to be considered as outbreak
perc_strata = 0.75 # proportion of data to be used for training

# SMOTE RELATED PARAMS
K_neighbors = 5 # number of nearest neighbors
Ratio_testtrain = 0 # for 50/50 outbreak/non-outbreak

In [26]:
drop_redundant_yn_features <- function(df) {
  # Get all column names
  cols <- names(df)
  
  # Find Yes/No pairs by stripping the suffix
  yes_cols <- cols[grepl("_yes", cols)]
  no_cols  <- cols[grepl("_no",  cols)]
  
  # Get base names for each
  yes_bases <- sub("_diff$", "", sub("_yes", "", yes_cols))
  no_bases  <- sub("_diff$", "", sub("_no",  "", no_cols))
  
  # Find bases that have BOTH a Yes and No column
  paired_bases <- intersect(yes_bases, no_bases)
  
  cols_to_drop <- c()
  
  for (base in paired_bases) {
    yes_col <- yes_cols[yes_bases == base]
    no_col  <- no_cols[no_bases == base]
    
    # Count NAs in each
    yes_nas <- sum(is.na(df[[yes_col]]))
    no_nas  <- sum(is.na(df[[no_col]]))
    
    # Drop whichever has more NAs 
    # If tied drop "No" (keep "Yes")
    if (no_nas >= yes_nas) {
      cols_to_drop <- c(cols_to_drop, no_col)
    } else {
      cols_to_drop <- c(cols_to_drop, yes_col)
    }
  }
  
  cat("Dropping", length(cols_to_drop), "redundant columns:\n")
  #cat(paste(" ", cols_to_drop), sep = "\n")
  
  df[, !names(df) %in% cols_to_drop]
}

set.seed(100)

df <- read_csv("data/merged_with_svi.csv", show_col_types = FALSE) |> clean_names() 
#df <- read_csv("data/merged_with_svi.csv") |> clean_names()

# outbreak >=2 is with outbreak, vice versa.
# df$outbreak <- as.integer(df$outbreak >= def_outbreak)

# SVI specific, because it does not include the data for Mcculloch, Mclennan, Mcmullen and Dewitt, 
# if we continue the prev way of data processing it will remove all SVI related features, so we remove these 4 counties.
#df <- df[!df$county %in% c("Mcculloch", "Mclennan", "Mcmullen", "Dewitt"), ]

df <- df %>% filter(!county %in% c("Mcculloch", "Mclennan", "Mcmullen", "Dewitt", "Loving"))
df_raw <- df
df$density <- df$population*1.0/df$area_sqmi
df <- df %>% select(-c("county", "phr", "area_sqmi"), -starts_with("m_"), -starts_with("mp_"), -starts_with("e_"), -starts_with("epl_"), -starts_with("spl_"), -starts_with("rpl_"), -starts_with("f_"))
df <- df %>% select(-c("ep_minrty", "ep_hisp", "ep_afam", "ep_pov150", "ep_uninsur"))
df <- df[,colSums(is.na(df)) == 0]

df <- drop_redundant_yn_features(df)

# split the data with stratified sampling
# strata <- ifelse(df$outbreak > 0, "nonzero", "zero")
index <- createDataPartition(df$outbreak, p = perc_strata, list = FALSE)
train <- df[index,]
test <- df[-index,]


#train <- train[, colSums(is.na(train)) == 0]
#train <- train[, !names(train) %in% c("county")]

# Classify as outbreak of not
train$outbreak  <- as.integer(train$outbreak >= def_outbreak)

set.seed(42)
# K for controlling the neighborhood, dup_size for ratio to achieve

smote_output <- SMOTE(
  X      = train[, names(train) != "outbreak"],
  target = train$outbreak,
  K      = K_neighbors, 
  dup_size = Ratio_testtrain
)

train_balanced <- smote_output$data
# SMOTE put target data in "class" col and rename it
names(train_balanced)[names(train_balanced) == "class"] <- "outbreak"
train_balanced$outbreak <- as.factor(train_balanced$outbreak)
cat("\nClass balance after SMOTE:\n")
print(table(train_balanced$outbreak))
head(train_balanced)

# mirror on test set
test <- test[, names(train)]
test <- test[, !names(test) %in% c("county")]
test$outbreak <- as.integer(test$outbreak >= def_outbreak)


Dropping 82 redundant columns:

Class balance after SMOTE:

  0   1 
177 170 


cve,enrollment,population,pct_hispanic,pct_black,pct_white,pct_poverty,pct_uninsured,pct_college,pct_foreign_born,⋯,ep_noveh,ep_groupq,ep_noint,ep_asian,ep_aian,ep_nhpi,ep_twomore,ep_otherrace,density,outbreak
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
4.55,666,2596,60.2,4.9,42.7,23.2,27.2,8.2,1.1,⋯,6.0,6.2,23.1,0.0,0.0,0.0,0.8,0.1,3.349323,1
14.54,3764,23167,40.3,2.1,70.2,4.5,36.7,11.7,1.4,⋯,4.1,0.7,18.3,0.9,0.5,0.0,0.6,0.2,15.420380,1
5.00,7416,44174,10.4,7.2,79.3,10.1,13.8,18.2,0.9,⋯,3.9,1.2,13.0,0.5,0.1,0.1,3.3,0.2,75.772807,1
2.42,33445,184344,46.0,7.2,55.9,8.9,14.5,31.0,1.1,⋯,4.7,1.1,11.6,1.9,0.2,0.0,2.1,0.1,204.745409,1
4.08,2117,11655,55.8,6.7,49.5,16.9,22.5,12.7,0.4,⋯,8.3,14.5,16.2,0.2,0.3,0.0,2.0,0.0,12.945605,1
4.12,5523,39397,13.1,5.9,81.2,9.1,14.3,19.5,0.7,⋯,3.2,8.6,19.7,0.4,0.4,0.1,3.4,0.1,44.224838,1


In [27]:
train_LR <- glm(outbreak ~ ., data = train_balanced, family = binomial)
train_LR_back <- step(train_LR,trace=0, direction = "both")

Warning message:
“glm.fit: algorithm did not converge”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: algorithm did not converge”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: algorithm did not converge”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: algorithm did not converge”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: algorithm did not converge”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: algorithm did not converge”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: algorithm did not converge”

In [28]:
probs   <- predict(train_LR_back, newdata=train, type="response")
summary(probs)   # check the range
lr_pred <- ifelse(probs > 0.5, 1, 0)
y_train <- as.numeric(as.character(train$outbreak))
table(Actual=as.factor(y_train), Predicted=as.factor(lr_pred))

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
0.00000 0.00000 0.00000 0.05348 0.00000 1.00000 

      Predicted
Actual   0   1
     0 177   0
     1   0  10

In [29]:
probs   <- predict(train_LR_back, newdata=test, type="response")
summary(probs)   # check the range
lr_pred <- ifelse(probs > 0.5, 1, 0)
y_test <- as.numeric(as.character(test$outbreak))
table(Actual=as.factor(y_test), Predicted=as.factor(lr_pred))

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.0000  0.0000  0.0000  0.1774  0.0000  1.0000 

      Predicted
Actual  0  1
     0 50  7
     1  1  4

In [30]:
probs   <- predict(train_LR_back, newdata=test, type="response")
summary(probs)   # check the range
lr_pred <- ifelse(probs > 0.5, 1, 0)
y_test  <- as.numeric(as.character(test$outbreak))
table(Actual=as.factor(y_test), Predicted=as.factor(lr_pred))

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.0000  0.0000  0.0000  0.1774  0.0000  1.0000 

      Predicted
Actual  0  1
     0 50  7
     1  1  4